# Hybrid Model v2 Training (Google Colab)

This notebook runs `hybrid_model_v2/train.py` using the pinned Colab-compatible setup from `docs/colab_compatibility.md`.

In [ ]:
# Runtime check
import os
import torch

print('COLAB_GPU:', os.environ.get('COLAB_GPU'))
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Pinned refs from docs/colab_compatibility.md
ACCELERATED_FEATURES_REF = 'e92685f57f8318b18725c5c8c0bd28c7fe188d9a'
SUPERPOINT_REF = '1411bbd68c50163555d39c1b26e9e046ebd48f27'
HYBRID_REPO_URL = 'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git'
HYBRID_REPO_REF = os.environ.get('HYBRID_REPO_REF', '')  # optional branch/commit pin
DATA_ROOT = '/content/MegaDepth'

print('ACCELERATED_FEATURES_REF =', ACCELERATED_FEATURES_REF)
print('SUPERPOINT_REF =', SUPERPOINT_REF)
print('HYBRID_REPO_REF =', HYBRID_REPO_REF or '<current clone default>')
print('DATA_ROOT =', DATA_ROOT)


In [ ]:
# Clone hybrid repo and install pinned Colab dependencies
%cd /content
!rm -rf XFeat-SuperPoint-Hybrid-Model
!git clone {HYBRID_REPO_URL}
%cd /content/XFeat-SuperPoint-Hybrid-Model
if HYBRID_REPO_REF:
    !git checkout {HYBRID_REPO_REF}
!pip -q install --upgrade pip
!pip -q install -r requirements-colab.txt


In [ ]:
# Clone and pin external repos required by the hybrid model
%cd /content
!rm -rf accelerated_features SuperPoint
!git clone https://github.com/verlab/accelerated_features.git
%cd /content/accelerated_features
!git checkout {ACCELERATED_FEATURES_REF}

%cd /content
!git clone https://github.com/rpautrat/SuperPoint.git
%cd /content/SuperPoint
!git checkout {SUPERPOINT_REF}


In [ ]:
# Ensure PYTHONPATH and download model weights
import os

hybrid_root = '/content/XFeat-SuperPoint-Hybrid-Model'
os.environ['PYTHONPATH'] = f"{hybrid_root}:/content/accelerated_features:/content/SuperPoint:{os.environ.get('PYTHONPATH','')}"

%cd /content/XFeat-SuperPoint-Hybrid-Model
!mkdir -p weights
!wget -q -O weights/xfeat.pt https://github.com/verlab/accelerated_features/releases/download/v1.0/xfeat.pt
!wget -q -O weights/superpoint_v1.pth https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth

print('PYTHONPATH head:', os.environ['PYTHONPATH'].split(':')[:3])


## Dataset layout required by `hybrid_model_v2/config.yaml`

Expected under `DATA_ROOT` (`/content/MegaDepth` by default):

- train scenes: `0001`, `0002`
- val scene: `0003`
- each scene contains `dense0/imgs` and `dense0/depths`


In [ ]:
# Optional: quick check for required scene folders
from pathlib import Path

required = ['0001', '0002', '0003']
for scene in required:
    imgs = Path(DATA_ROOT) / scene / 'dense0' / 'imgs'
    depths = Path(DATA_ROOT) / scene / 'dense0' / 'depths'
    print(scene, 'imgs:', imgs.exists(), 'depths:', depths.exists())


In [ ]:
# Start training
%cd /content/XFeat-SuperPoint-Hybrid-Model
!python hybrid_model_v2/train.py --config hybrid_model_v2/config.yaml --data_root {DATA_ROOT}
